# 1교시 · VLM 경량화 개관 · 양자화 기초

> **VLM 경량화 2일 과정 · Day 1 (1교시) · 이론**
> 실습 환경: **Google Colab (T4 GPU)** · 모델: **Qwen/Qwen3-VL-4B-Instruct** · 데이터: **HuggingFaceM4/ChartQA**

---

## 이 교시의 목표
- 왜 **경량화**가 필요한지(메모리·비용·배포)와 **양자화**의 기본 원리를 이해한다.
- 정밀도(fp16/bf16/int8/int4)와 비트당 메모리를 **직접 계산**해 "왜 4bit인가"를 수치로 안다.
- **NF4 · 블록 단위 양자화 · 이중 양자화 · QLoRA**의 개념을 잡는다.
- **PTQ vs QAT vs QLoRA**와 **AWQ**의 위치를 구분하고 2일 로드맵을 본다.

---

### 📌 이 과정 한 줄 요약
**Day 1 = 학습을 가볍게(QLoRA)**, **Day 2 = 배포를 가볍게(AWQ + vLLM 서빙)**.
같은 데이터(**ChartQA**)로 이틀을 관통하며, 마지막에 *학습→병합→양자화→서빙* 전체 파이프라인으로 잇습니다.

| | Day 1 (T4) | Day 2 (A100 80GB) |
|---|---|---|
| 목표 | **학습 경량화** | **배포 경량화** |
| 기법 | **QLoRA** (bitsandbytes 4bit) | **AWQ** (llm-compressor) + **vLLM 서빙** |
| 모델 | Qwen3-VL-4B (도전 8B) | Day1 병합 4B + 베이스 8B |


## 0. 공통 헤더 — Google Drive(VLM_FT_2) 마운트 + HF_TOKEN 로드

> 📌 **모든 Day 1 노트북은 이 셀을 먼저 실행합니다.** Google Drive의 작업 폴더 **`VLM_FT_2`** 를 마운트하고, `.env`의 **HF_TOKEN**을 불러옵니다.
> - `VLM_DIR` / `DATA_DIR` : 교시 간 공유 폴더(전처리 데이터·어댑터·결과가 여기 모입니다).
> - **HF_TOKEN**: `VLM_FT_2/.env` 에 `HF_TOKEN=hf_...` 한 줄을 넣어두면 자동 로드됩니다(다운로드 경고 방지·비공개 모델 접근). `login()`은 호출하지 않습니다(환경변수만으로 충분, 경고 방지).

In [3]:
# ════════════════════════════════════════════════════════════
#  공통 헤더 · Google Drive(VLM_FT_2) 마운트 + HF_TOKEN(.env) 로드
#  (모든 Day1 노트북 상단에서 동일하게 실행)
# ════════════════════════════════════════════════════════════
import os

# (1) Google Drive 마운트 → 작업 폴더 VLM_FT_2 (교시 간 데이터·어댑터·결과 공유)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    VLM_DIR = '/content/drive/MyDrive/VLM_FT_2'      # Drive 경로(권장)
except Exception:
    VLM_DIR = '/content/VLM_FT_2'                     # Colab 아니면 로컬 폴백
DATA_DIR = f'{VLM_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)

# (2) .env 에서 HF_TOKEN 로드. login()은 부르지 않음(환경변수만으로 인증, 경고 없음).
try:
    from dotenv import load_dotenv
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
    from dotenv import load_dotenv
ENV_PATH = f'{VLM_DIR}/.env'
if os.path.exists(ENV_PATH):
    load_dotenv(ENV_PATH)
    print('HF_TOKEN:', '로드됨' if os.environ.get('HF_TOKEN') else '없음')
else:
    print(f'.env 없음 — {ENV_PATH} 에 HF_TOKEN=hf_... 한 줄을 만들면 자동 로드됩니다(공개 데이터만 쓰면 없어도 됨)')
print('작업 폴더 VLM_DIR =', VLM_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HF_TOKEN: 로드됨
작업 폴더 VLM_DIR = /content/drive/MyDrive/VLM_FT_2


## 0-B. 선수 과정 복습 — VLM은 이미지를 어떻게 '읽는가' + LoRA

> 🧑‍🏫 **선수 과정(VLM 파인튜닝)을 안 들었거나 복습이 덜 된 분을 위한 압축 복습입니다.** 이미 익숙하면 건너뛰어도 됩니다. 여기 나오는 용어(이미지 토큰·비전 타워·LoRA)는 이 과정 내내 쓰입니다.

**VLM의 3단계 구조** (Qwen3-VL 기준)
1. **비전 인코더(ViT)** — 이미지를 작은 **패치**(이 모델은 16×16픽셀)로 쪼개 각 패치를 벡터로 인코딩합니다. 이 부분을 흔히 **비전 타워**라고 부릅니다.
2. **Connector / Merger** — 인접한 **2×2 패치를 하나로 묶어** LLM이 받을 **'이미지 토큰'** 으로 압축합니다(토큰 수 1/4로 감소).
3. **LLM(언어모델)** — `이미지 토큰 + 텍스트 토큰`을 **하나의 시퀀스**로 받아 답을 생성합니다. 즉 모델 입장에서 이미지는 '특수한 토큰 묶음'일 뿐입니다.

> **DeepStack** — Qwen3-VL은 ViT 여러 층의 특징을 LLM 여러 층에 나눠 주입합니다. 그래서 작은 글자·차트·표 같은 **조밀한 시각 정보에 강합니다**(ChartQA에 유리).

**왜 '경량화'에서 이 구조가 중요한가** — 두 과정 공통 원칙은 *"비전 타워는 가급적 건드리지 않는다"* 입니다.
- **Day1(QLoRA)**: 학습 때 **비전 타워를 동결**합니다(메모리↓·안정성↑). LoRA도 LLM 선형층에만 붙입니다.
- **Day2(AWQ)**: 양자화에서 **`visual` 모듈을 제외**(ignore)합니다 — 비전 특징이 망가지면 화질 인식이 무너지기 때문입니다.

In [4]:
# 이미지 해상도(w, h)로부터 '이미지 토큰 수'를 개념적으로 추정하는 함수
def estimate_visual_tokens(width, height, patch_size=16, merge_size=2):
    # 1) ViT는 이미지를 patch_size x patch_size 패치로 분할
    # 2) Qwen3-VL은 2x2 패치를 병합해 최종 이미지 토큰을 만듦
    #    -> 토큰 1개가 대표하는 한 변의 픽셀 수 = patch_size * merge_size (기본 32px)
    eff = patch_size * merge_size

    # 가로/세로 토큰 개수(개념적 근사치)
    # round를 써서 대략적인 토큰 격자 크기를 계산
    cols = max(1, round(width / eff))
    rows = max(1, round(height / eff))

    # ViT 패치 총개수(정수 나눗셈 기준, 단순 계산)
    patches = (width // patch_size) * (height // patch_size)

    # 병합 후 이미지 토큰 총개수(대략 패치의 1/4 수준)
    tokens = cols * rows

    # (패치 수, 토큰 수, (가로 토큰, 세로 토큰)) 반환
    return patches, tokens, (cols, rows)


# 예시 해상도 3개에 대해 패치/토큰 수 출력
for (w, h) in [(448, 448), (896, 672), (1288, 952)]:
    patches, tokens, grid = estimate_visual_tokens(w, h)
    print(f"{w}x{h}: 패치 약 {patches}개 -> 이미지 토큰 약 {tokens}개 (격자 {grid[0]}x{grid[1]})")

# ✏️ 미니 실습 정답 힌트:
# 해상도 가로·세로를 각각 2배로 키우면 면적이 4배가 되므로,
# 이미지 토큰 수도 보통 '약 4배' 증가합니다.

448x448: 패치 약 784개 -> 이미지 토큰 약 196개 (격자 14x14)
896x672: 패치 약 2352개 -> 이미지 토큰 약 588개 (격자 28x21)
1288x952: 패치 약 4720개 -> 이미지 토큰 약 1200개 (격자 40x30)


### 🔍 결과 해석 — 이미지 한 장 = 토큰 수백 개

- `448x448 → 이미지 토큰 약 196개`, 해상도가 커질수록 토큰이 **제곱으로** 늘어납니다.
- 즉 **이미지 한 장이 텍스트 수백 단어와 맞먹는** 시퀀스 길이를 차지합니다. 이것이 뒤(5-2·5-4절)에서 다룰 **활성값 메모리 폭증**의 핵심 원인이고, 3교시에서 이미지를 `shrink(768)`로 줄이는 이유입니다.

### 🔭 비전 인코더와 LLM은 '차원'이 다르다 — 어떻게 잇는가

방금 본 *이미지 한 장 = 토큰 수백 개*를 **차원(dimension) 관점**에서 한 번 더 봅니다. 핵심은 **"비전 인코더가 쓰는 벡터 차원 ≠ LLM이 쓰는 벡터 차원"** 이고, 그 사이를 **merger의 projection**이 잇는다는 점입니다. (숫자는 **Qwen3-VL-4B 실제 config** 기준)

**픽셀 → 패치 → 이미지 토큰, 단계별로**
1. **픽셀 → 패치**: 이미지를 **16×16 픽셀** 패치로 자릅니다(이 모델 `patch_size=16`). 448×448면 `448/16 = 28` → **28×28 = 784개 패치**.
2. **패치 → ViT 임베딩**: 비전 인코더(ViT, 24층)가 각 패치를 **1024차원** 벡터로 인코딩합니다(`vision hidden_size=1024`).
3. **2×2 병합(merger) → 이미지 토큰**: 인접 **2×2 패치 4개를 이어붙여**(`1024×4 = 4096`) 작은 MLP가 **2560차원**으로 투영합니다(`out_hidden_size=2560`). 토큰 수는 1/4인 **14×14 = 196개**, 각 **2560차원**.
4. **LLM에 합류**: LLM의 텍스트 토큰 임베딩도 **2560차원**(`text hidden_size=2560`). 이미지 토큰이 **같은 2560차원**이라 `<|image_pad|>` 자리(3교시 `mm_token_type_ids=1` cf:text type=0)에 그대로 끼워져 텍스트와 **한 시퀀스**가 됩니다.

| 단계 | 개수(448×448 예) | 벡터 차원 |
|---|---|---|
| 패치(16×16) | 784 | 16×16 px · 3채널 |
| ViT 패치 임베딩 | 784 | **1024** (비전 인코더 내부) |
| 2×2 병합 직후(이어붙임) | 196 | 1024×4 = **4096** |
| **projection 후 = 이미지 토큰** | **196** | **2560** (LLM과 동일) |
| 텍스트 토큰 임베딩 | (단어 수만큼) | **2560** |

**왜 '차원 맞추기'가 필요한가** — ViT는 **1024차원**으로, LLM은 **2560차원**으로 일합니다. 둘은 그대로 못 더합니다. **merger의 projection(4096 → 2560)** 이 바로 *"비전 언어를 LLM 언어로 번역하는 어댑터"* 입니다. Qwen3-VL은 비전의 `out_hidden_size` 를 LLM `hidden_size` 와 **같은 2560으로 설계**해 둘이 맞물리게 했습니다.

> **M-RoPE 한 스푼**: 합쳐진 시퀀스의 위치 정보는 `text_config` 의 `mrope_section = [24, 20, 20]` 으로 쪼개집니다 — head 차원의 절반(128/2 = 64)을 **시간(t)·세로(h)·가로(w)** 에 `24/20/20` 으로 배분. 그래서 이미지 토큰은 `(t, h, w)` 좌표(3교시 `image_grid_thw`)를, 텍스트 토큰은 1D 위치를 받습니다.

**경량화와의 연결** — 이 구조 때문에 *"비전 타워(ViT·1024차원부)는 건드리지 않는다"* 가 자연스럽습니다. Day1 QLoRA는 LLM 선형층(2560차원 쪽)에만 LoRA를 붙이고, Day2 AWQ는 `visual` 모듈을 양자화에서 제외합니다 — **번역기(merger)와 비전 특징이 망가지면 LLM이 멀쩡해도 "그림을 잘못 읽기"** 때문입니다.

### LoRA 복습 — QLoRA의 절반

- **전체 파인튜닝**은 모델의 모든 가중치 `W`를 업데이트 → 옵티마이저 상태까지 더해 메모리가 큽니다(5-2절).
- **LoRA**: 원본 `W`는 **얼리고**, 작은 두 행렬 `A·B`(저랭크 `r`)만 학습해 `ΔW = B·A`로 근사합니다. 학습 파라미터가 **전체의 1% 미만**으로 줄어듭니다.
- 붙이는 위치(`target_modules`)는 LLM 선형층 — 어텐션 `q_proj·k_proj·v_proj·o_proj` 와 MLP `gate_proj·up_proj·down_proj`. (비전 타워엔 붙이지 않습니다.)

> 💡 **LoRA + 4bit 양자화 = QLoRA.** 이 노트북 뒤에서 다루는 **4bit(NF4)** 가 나머지 절반입니다. 4교시에서 둘을 합쳐 T4 한 장으로 4B 모델을 파인튜닝합니다.

## 1. 왜 경량화인가?

VLM은 무겁습니다. 8B 모델은 가중치만 **fp16로 약 17.5GB** — T4(16GB)엔 **그대로 올라가지도 않습니다**.
경량화의 세 가지 동기:

- **메모리**: 더 작은 GPU(T4)에서 더 큰 모델을 **학습·추론**.
- **비용·속도**: 4bit로 줄이면 메모리 대역폭이 줄어 **추론이 빨라지고** 서빙 비용이 내려감.
- **배포**: 엣지·온프레미스 등 **제한된 환경**에 올릴 수 있음.

경량화는 두 갈래입니다. 이 과정은 이 둘을 하루씩 다룹니다.

| 갈래 | 무엇을 줄이나 | 대표 기법 | 이 과정 |
|---|---|---|---|
| **학습 경량화** | 파인튜닝에 드는 메모리 | **QLoRA** | **Day 1** |
| **배포 경량화** | 추론/서빙에 드는 메모리·지연 | **AWQ** | **Day 2** |

## 2. 정밀도(precision)와 비트

숫자 하나를 **몇 비트**로 저장하느냐가 메모리를 결정합니다.

| 타입 | 비트 | 1B 파라미터당 | 특징 |
|---|---|---|---|
| fp32 | 32 | 4 GB | 전체 정밀(무거움) |
| **fp16 / bf16** | 16 | 2 GB | VLM 추론·학습 표준 |
| **int8** | 8 | 1 GB | 경량 추론 |
| **int4 (nf4 등)** | 4 | 0.5 GB | 공격적 경량화(QLoRA·AWQ) |

> **핵심 직관**: 가중치 메모리 ≈ **파라미터 수 × (비트/8)**. 비트를 16→4로 줄이면 **메모리가 1/4**.
> bf16은 fp16보다 표현 범위가 넓어 학습이 안정적입니다 — **A100은 bf16**, **T4는 fp16**을 씁니다(T4는 bf16 미지원).

**참고**  
- Ampere Architecture (nvidia GPU) - 저장은 fp16으로 연산은 
- BF16(BFloat16, Brain Floating Point)은 인공지능(AI) 및 딥러닝 모델에 최적화된 16비트 부동소수점 데이터 형식입니다. 구글 브레인(Google Brain) 팀에서 개발했으며, 최근 ChatGPT나 LLM 같은 거대 AI 모델을 학습하고 실행할 때 필수적으로 사용

## 3. 양자화(quantization)란?

**연속적인 실수(fp16) 가중치를 적은 단계의 정수로 매핑**하는 것입니다.

- 실수 범위를 **scale**(과 zero-point)로 정수 격자에 대응: `q = round(w / scale)`, 복원은 `w ≈ q × scale`.
- 한 텐서를 통째로 양자화하면 **이상치(outlier)** 때문에 손실이 큽니다 → **블록 단위(group-wise)** 로 작은 묶음(예: 64·128개)마다 scale을 따로 둡니다.
- **잃는 것**: 정밀도(반올림 오차). **얻는 것**: 메모리·속도.

```
  fp16 가중치:  ... 0.13  -0.07   0.91  -1.20  0.04 ...   (실수, 16bit)
                       │ round(w/scale)
                       ▼
  int4 격자:    ...   2     -1      7    -8     1  ...    (정수, 4bit)
```

> 블록 단위 양자화: 128개씩 묶어 묶음마다 scale을 따로 두면, 한 묶음의 이상치가 **다른 묶음을 오염시키지 않습니다**.

## 4. NF4 · 이중 양자화 · QLoRA

Day 1에서 쓸 **QLoRA**는 세 가지 아이디어의 결합입니다.

**① NF4 (NormalFloat 4-bit)**
신경망 가중치는 대체로 **0 근처 정규분포**를 따릅니다. NF4는 일반 int4처럼 균등 간격이 아니라 **정규분포에 맞춰 격자 간격을 배치**해, 같은 4bit로도 손실을 줄입니다.

**② 이중 양자화 (Double Quantization)**
블록마다 두는 **scale 값들 자체도 양자화**합니다. 파라미터당 약 0.5bit를 추가로 절약합니다.

**③ QLoRA**
base 모델을 **4bit로 동결**해 메모리를 확 줄이고, 그 위에 **작은 LoRA 어댑터(fp16)만 학습**합니다. 즉 *무거운 부분은 4bit로 고정, 학습은 가벼운 어댑터에서*.

```
  [ 4bit 동결 base (학습 안 함) ]  ←  메모리 대부분 차지하지만 4bit라 작음
            +
  [ LoRA 어댑터 (fp16, 학습함) ]  ←  파라미터의 1% 미만, 여기만 그래디언트
```

> 그래서 QLoRA는 **T4 같은 작은 GPU에서도 파인튜닝**이 됩니다. Day1-4에서 직접 돌립니다.

## 5. 메모리 수학 — 직접 계산해보기

아래 셀은 **순수 파이썬**이라 GPU 없이도 즉시 실행됩니다. Qwen3-VL의 실제 파라미터 수로 정밀도별 **가중치 메모리**를 계산하고, **T4(16GB)** 에 올라가는지 판정합니다.

In [3]:
# ── 정밀도별 가중치 메모리 추정 (순수 파이썬, GPU 불필요) ──────────────────
# 핵심 공식:
#   가중치 메모리(GB) ≈ 파라미터 수 × (비트 / 8) / 1024^3
# 예)
#   8B 모델을 fp16(16bit)로 로드하면 대략 16GB
#   같은 모델을 int4로 로드하면 대략 4GB (1/4)

# T4는 총 16GB지만, 런타임/버퍼 여유를 위해 14GB를 실사용 한계로 가정
USABLE_T4_LIMIT_GB = 14.0
GiB = 1024 ** 3

# 비교할 모델(검증된 파라미터 수)과 정밀도(비트)  ← 표의 행/열을 이룸
PARAMS = {'Qwen3-VL-4B': 4_437_800_000, 'Qwen3-VL-8B': 8_767_100_000}
BITS   = {'fp16/bf16': 16, 'int8': 8, 'int4(nf4)': 4}

def weight_mem_gb(num_params: int, bits: int) -> float:
    """
    파라미터 개수와 비트 정밀도로 가중치 메모리(GB)를 계산.
    - num_params: 파라미터 수(개)
    - bits: 파라미터당 비트 수 (fp16=16, int8=8, int4=4)
    """
    bytes_total = num_params * (bits / 8)   # 총 바이트
    return bytes_total / GiB                # GiB 단위로 변환

# 출력 표 헤더
header = f"{'모델':<14}{'정밀도':<12}{'가중치(GB)':>12}{'T4 적재':>10}"
print(header)
print('-' * len(header))

# 모델별 × 정밀도별 메모리 계산
for model, n in PARAMS.items():
    for precision_name, precision_bits in BITS.items():
        gb = weight_mem_gb(n, precision_bits)

        # 14GB 미만이면 적재 가능(OK), 이상이면 초과(X)
        fits = 'OK' if gb < USABLE_T4_LIMIT_GB else 'X (초과)'

        # 소수점 둘째 자리까지 출력
        print(f"{model:<14}{precision_name:<12}{gb:>10.2f} GB{fits:>10}")
    print('-' * len(header))


모델            정밀도              가중치(GB)     T4 적재
------------------------------------------------
Qwen3-VL-4B   fp16/bf16         8.27 GB        OK
Qwen3-VL-4B   int8              4.13 GB        OK
Qwen3-VL-4B   int4(nf4)         2.07 GB        OK
------------------------------------------------
Qwen3-VL-8B   fp16/bf16        16.33 GB    X (초과)
Qwen3-VL-8B   int8              8.16 GB        OK
Qwen3-VL-8B   int4(nf4)         4.08 GB        OK
------------------------------------------------


## 5-1. 실행 결과 읽기 — 표가 말하는 것

방금 출력된 표를 한 줄씩 해석해 봅시다.

| 모델 | 정밀도 | 가중치 | T4 적재 |
|---|---|---|---|
| 4B | fp16/bf16 | **8.27 GB** | OK |
| 4B | int8 | 4.13 GB | OK |
| 4B | int4(nf4) | **2.07 GB** | OK |
| 8B | fp16/bf16 | **16.33 GB** | **X (초과)** |
| 8B | int8 | 8.16 GB | OK |
| 8B | int4(nf4) | **4.08 GB** | OK |

**관찰 1 — 정밀도를 반으로 줄이면 메모리도 반.** 같은 모델에서 fp16→int8→int4로 갈 때 가중치가 정확히 **절반씩**(8.27→4.13→2.07) 줄어듭니다. *가중치 메모리 = 파라미터 수 × (비트/8)* 이라는 공식 그대로입니다.

**관찰 2 — 8B fp16만 유일하게 'X'.** 8B를 fp16으로 올리면 **16.33 GB**로, T4의 총 VRAM **16 GB**를 넘습니다. 표의 판정 기준을 **14 GB**로 둔 이유는, GPU에는 가중치 말고도 **CUDA 컨텍스트·중간 버퍼** 등이 1~2 GB 상주하기 때문입니다. 즉 "가중치가 총량에 가까우면 실제로는 이미 터진다"를 반영한 보수적 한계선입니다.

**관찰 3 — 4bit가 8B를 살린다.** 같은 8B라도 int4로 내리면 **4.08 GB** — T4에 넉넉히 들어갑니다. *큰 모델을 작은 GPU에 올리는 가장 직접적인 수단이 4bit*임을 보여줍니다.

> 정리하면, **추론**(가중치만 올리면 되는 경우)은 이 표대로면 8B도 4bit로 T4에서 가능합니다. 문제는 **학습**입니다. 학습은 가중치 외에 더 많은 것이 필요하거든요. 그게 다음 절입니다.

## 5-2. 학습은 가중치만으로 끝나지 않는다 — 학습 메모리의 4대 구성

위 표는 **가중치**만 따졌습니다. 하지만 **파인튜닝(학습)** 을 하려면 가중치 위에 다음이 더 얹힙니다.

> 📌 **표기 약속**: 아래의 **'N바이트/param'** 은 *파라미터 1개를 저장하는 데 드는 바이트 수*입니다. 바이트 수는 **정밀도 ÷ 8**로 정해집니다 — fp16=16비트=**2바이트**, fp32=32비트=**4바이트**, int8=8비트=**1바이트**. (참고: "**4B 모델**"의 B는 Billion(10억)이라 여기서 말하는 바이트와는 전혀 다른 뜻입니다.)

**① 가중치 (weights)**
모델 파라미터 그 자체. 추론에도 필요한 부분입니다. 혼합정밀 학습에선 보통 **fp16 사본(2바이트/param)** 을 씁니다.

**② 그래디언트 (gradients)**
역전파로 각 **학습 대상** 파라미터마다 계산되는 기울기. 파라미터와 **같은 개수**만큼 생깁니다(fp16이면 2바이트/param). *학습하는 파라미터에만* 붙는다는 점이 핵심입니다.

**③ 옵티마이저 상태 (optimizer states)**
Adam 계열은 파라미터마다 **모멘텀(1차)** 과 **분산(2차)** 두 통계를 들고 다닙니다. 수치 안정성을 위해 보통 **fp32(각 4바이트/param)** → 합쳐서 **8바이트/param**. 게다가 혼합정밀이면 **fp32 마스터 가중치(4바이트/param)** 까지 둡니다. **학습 메모리의 가장 큰 덩어리**가 여기입니다.
-  모멘텀 그레디언트 두 가지 *2  

**④ 활성값 (activations)**
순전파 중간 출력들. 역전파에서 기울기를 구하려면 저장해 둬야 합니다. **배치 크기·시퀀스 길이·레이어 수**에 비례해 커지며, VLM은 이미지 토큰까지 더해져 만만치 않습니다. **gradient checkpointing**(중간값을 저장 대신 역전파 때 재계산)으로 크게 줄일 수 있습니다.

---

### 어림셈 — "전체 파인튜닝 ≈ 16바이트/param"의 16은 어디서?

Adam + 혼합정밀 **전체 파인튜닝**은 파라미터당 약 **16바이트**가 듭니다(활성값 제외). 이 16은 위 네 구성을 **바이트로 환산해 더한 값**입니다. 각 항목의 숫자는 **정밀도 ÷ 8**(비트→바이트)에서 나옵니다.

| 항목 | 정밀도 | 바이트/param | 
|---|---|---|
| fp16 가중치 사본 | fp16 (16비트) | **2** | 
| fp16 그래디언트 | fp16 (16비트) | **2** | 
| fp32 옵티마 모멘텀 | fp32 (32비트) | **4** | 
| fp32 옵티마 분산 | fp32 (32비트) | **4** | 
| fp32 마스터 가중치 | fp32 (32비트) | **4** | 
| **합계** | | **16** | 

> 즉 수식 `2 + 2 + 4 + 4 + 4 = 16` 에서 **2는 fp16(16비트)**, **4는 fp32(32비트)** 를 바이트로 바꾼 값입니다. 정밀도가 높을수록(비트↑) 항목당 바이트가 커집니다.
> 결과적으로 학습은 *추론용 가중치(2바이트/param)의 약 8배*. 아래에서 4B로 직접 계산해 확인합니다.

In [4]:
# ── 학습 메모리 구성 분해 (순수 파이썬, 활성값 제외) ──────────
GiB = 1024 ** 3
P = 4_437_800_000   # Qwen3-VL-4B 파라미터 수

def gib(b): 
    """바이트를 GiB 단위로 변환"""
    return b / GiB

# (A) 전체 파인튜닝: 모든 파라미터를 학습 (Adam + 혼합정밀)
#     각 숫자 = 정밀도 ÷ 8 (비트→바이트):  fp16→2바이트,  fp32→4바이트
full = {
    'fp16 가중치 사본':   P * 2,   # 2바이트/param - 모델 가중치 원본
    'fp16 그래디언트':    P * 2,   # 2바이트/param - 역전파로 계산된 기울기
    'fp32 옵티마 모멘텀': P * 4,   # 4바이트/param - Adam의 1차 모멘텀 통계
    'fp32 옵티마 분산':   P * 4,   # 4바이트/param - Adam의 2차 분산 통계
    'fp32 마스터 가중치': P * 4,   # 4바이트/param - 혼합정밀 학습용 fp32 마스터 사본
}

# (B) QLoRA: base는 4bit로 동결(학습 X), 작은 LoRA 어댑터만 학습
LORA_RATIO = 0.005                 # 어댑터가 전체의 약 0.5% (r·타깃에 따라 변동)
P_adapter = int(P * LORA_RATIO)    # 어댑터 파라미터 개수 = 전체의 0.5%
qlora = {
    '4bit 동결 base (학습X)':  P * 0.5,        # 0.5바이트/param - NF4 4bit로 로드되고 학습 안 함
    'fp16 어댑터 가중치':      P_adapter * 2,  # 2바이트/param - LoRA 가중치 (학습 대상)
    'fp16 어댑터 그래디언트':  P_adapter * 2,  # 2바이트/param - LoRA의 기울기
    '8bit 옵티마 상태(2종)':   P_adapter * 2,  # paged_adamw_8bit로 압축된 옵티마이저 상태
}

def show(title, parts):
    """메모리 구성을 정렬하여 출력하는 헬퍼 함수"""
    print(title)
    for k, v in parts.items():
        print(f'  {k:<22}{gib(v):>8.2f} GiB')
    total = sum(parts.values())
    print(f'  {"─ 합계(활성값 제외)":<22}{gib(total):>8.2f} GiB')
    print(f'  (= 파라미터당 {total / P:.2f} 바이트)')
    return total

# [A] 전체 파인튜닝 메모리 계산 및 출력
full_total  = show('[A] 전체 파인튜닝 (모든 파라미터 학습)', full)
print()

# [B] QLoRA 메모리 계산 및 출력
qlora_total = show('[B] QLoRA (4bit 동결 base + 0.5% 어댑터)', qlora)
print()

# 학습 대상 파라미터 비교
print(f'학습 대상 파라미터: 전체FT {P/1e9:.2f}B  vs  QLoRA {P_adapter/1e6:.1f}M')

# 메모리 절감률 계산 및 출력
print(f'메모리 절감(활성값 제외): {gib(full_total):.1f} GiB → {gib(qlora_total):.1f} GiB '
      f'(약 {full_total / qlora_total:.0f}배)')
print()

# T4(16GB, 실사용 14GB 기준) 적재 가능 여부 판정
print(f'T4(16GB) 적재:  전체FT → {"불가" if gib(full_total) > 14 else "가능"}'
      f'   |  QLoRA → {"가능" if gib(qlora_total) < 14 else "불가"}')

[A] 전체 파인튜닝 (모든 파라미터 학습)
  fp16 가중치 사본               8.27 GiB
  fp16 그래디언트                8.27 GiB
  fp32 옵티마 모멘텀             16.53 GiB
  fp32 옵티마 분산              16.53 GiB
  fp32 마스터 가중치             16.53 GiB
  ─ 합계(활성값 제외)             66.13 GiB
  (= 파라미터당 16.00 바이트)

[B] QLoRA (4bit 동결 base + 0.5% 어댑터)
  4bit 동결 base (학습X)        2.07 GiB
  fp16 어댑터 가중치              0.04 GiB
  fp16 어댑터 그래디언트            0.04 GiB
  8bit 옵티마 상태(2종)           0.04 GiB
  ─ 합계(활성값 제외)              2.19 GiB
  (= 파라미터당 0.53 바이트)

학습 대상 파라미터: 전체FT 4.44B  vs  QLoRA 22.2M
메모리 절감(활성값 제외): 66.1 GiB → 2.2 GiB (약 30배)

T4(16GB) 적재:  전체FT → 불가   |  QLoRA → 가능


## 5-3. 분해 결과 해석 — 무엇이 메모리를 줄였나

### 출력 읽기 — `[A]`와 `[B]`를 한 줄씩

**`[A]` 전체 파인튜닝** (출력 그대로):
- `fp16 가중치 사본`·`fp16 그래디언트` 각 **8.27 GiB** = 4.44B × **2바이트**. *추론용 가중치와 똑같은 크기가 그래디언트로 한 번 더* 듭니다.
- `fp32 모멘텀`·`fp32 분산`·`fp32 마스터` 각 **16.53 GiB** = 4.44B × **4바이트**. **이 fp32 3종이 합계의 약 75%**(49.6/66.1) — fp16의 2배 크기라 셋을 합치면 가장 무겁습니다.
- 합계 **66.13 GiB = 파라미터당 16.00바이트** → 어림셈(2+2+4+4+4)과 **정확히 일치**.

**`[B]` QLoRA** (출력 그대로):
- `4bit 동결 base` **2.07 GiB**가 거의 전부. 하지만 **동결**이라 여기엔 그래디언트·옵티마이저가 **0**입니다.
- 어댑터 3줄(`가중치`·`그래디언트`·`8bit 옵티마`)은 각 **0.04 GiB**뿐. 학습 대상이 **22.2M(전체의 0.5%)** 라 미미합니다.
- 합계 **2.19 GiB = 파라미터당 0.53바이트** → `[A]` 대비 **약 30배** 작습니다.

> 한 문장 요약: **`[A]`에서 메모리를 키운 주범은 fp32 3종(모멘텀·분산·마스터)**, **`[B]`가 그걸 없앤 비결은 학습 대상을 22.2M로 줄인 것**(LoRA + base 동결)입니다.

---

### 무엇이 각 항목을 줄였나

이 절감은 **Day1-4에서 쓸 기법들이 각 항목을 공격**한 결과입니다.

| 줄인 대상 | 어떻게 | 효과 |
|---|---|---|
| 가중치 | **4bit(NF4) 로드** | base 가중치 8.3→2.1 GiB |
| 그래디언트·옵티마이저 | **LoRA**(0.5%만 학습) + **base 동결** | 학습 대상이 4.4B → 22M로 급감 |
| 옵티마이저 상태 | **`paged_adamw_8bit`** | 상태를 fp32(8바이트)→8bit(2바이트)로 |
| 활성값 | **gradient checkpointing** | 저장 대신 재계산(위 계산엔 미포함) |

> ⚠️ 위 합계는 **활성값을 뺀** 값입니다. 실제로는 활성값이 batch·시퀀스·이미지 토큰에 따라 수 GB 더해지지만, QLoRA는 grad checkpointing과 합쳐 그마저 T4 안에 욱여넣습니다. **실측 피크 VRAM은 Day1-4에서 직접 확인**합니다.

**한 줄 결론**: 추론은 *가중치*만의 게임이지만, 학습은 *그래디언트·옵티마이저·활성값*까지의 게임입니다. QLoRA는 이 네 항목을 동시에 깎아 "작은 GPU로 파인튜닝"을 가능하게 합니다.

## 5-4. 그런데 왜 '활성값 제외'였나 — 활성값을 따로 보는 이유

앞의 분해 합계에 줄곧 **(활성값 제외)** 라고 단서를 달았습니다. 활성값만 **성격이 다르기 때문**입니다.

- 가중치·그래디언트·옵티마이저 상태는 **파라미터 수에 정확히 비례**합니다. `파라미터 수 × 바이트/param` 한 줄로 떨어지고, **모델만 정해지면 값이 결정**됩니다.
- 반면 **활성값은 파라미터 수만으로 정해지지 않습니다.** 같은 모델이라도 *어떻게 학습을 돌리느냐*에 따라 수 배씩 달라집니다.

그래서 "파라미터당 N바이트" 같은 깔끔한 공식으로 못 적고, 어림셈을 흐리지 않으려고 분리해 둔 것입니다.

### 활성값을 좌우하는 요소

| 요소 | 영향 | 직관 |
|---|---|---|
| **배치 크기(batch)** | 비례 | batch 2배 → 활성값 약 2배 |
| **시퀀스 길이(seq)** | 비례 | 토큰이 길수록 저장할 중간값↑ |
| **이미지 토큰(VLM)** | seq에 더해짐 | 큰/고해상도 이미지 = 토큰 폭증 → seq↑ |
| **은닉 차원·레이어 수** | 비례 | 모델이 깊고 넓을수록↑ (모델 고정값) |
| **gradient checkpointing** | 크게 감소 | 중간값을 저장 대신 **역전파 때 재계산** |

> **VLM 특유의 포인트**: 텍스트 LLM과 달리, VLM은 **이미지가 토큰으로 환산**돼 seq에 더해집니다. 질문은 20토큰이어도 이미지 한 장이 수십~수백 토큰이 될 수 있어, *이미지 해상도가 곧 활성값 메모리*가 됩니다. Day1-3에서 `max_pixels`로 이미지 토큰 상한을 두는 이유가 바로 이것입니다.

아래 셀로 **batch·seq·체크포인팅**에 따라 활성값이 어떻게 변하는지 대략적으로 가늠해 봅니다.

In [5]:
# ── 활성값 메모리 스케일링 가늠 (대략적 추정) ────────────────
# 주의: 절대값이 아니라 '무엇에 비례하는지'를 보기 위한 근사입니다.
#       정확한 값은 모델 구조·구현에 따라 달라집니다.
# ── 모델 고정값(Qwen3-VL-4B) + 활성값 1단위 바이트 (이 셀만 돌려도 되도록 여기서 정의) ──
GiB    = 1024 ** 3      # 바이트 → GiB
HIDDEN = 2560           # 디코더 은닉 차원(text hidden_size)
LAYERS = 36             # 디코더 레이어 수(num_hidden_layers)
BYTES  = 2              # 활성값 dtype 바이트 (fp16 = 2)

def act_gib(batch, seq, checkpoint=True, M=16):
    """활성값 메모리(GiB) 근사.
    - 활성값 1단위 = batch × seq × hidden  (레이어 입력 한 장 분량)
      → batch: 한 번에 처리하는 샘플 수
      → seq  : 텍스트 + 이미지 토큰을 합친 시퀀스 길이
      → hidden: 디코더 은닉 차원 (Qwen3-VL-4B = 2560)
    - 체크포인팅 ON : 레이어 경계(입력)만 저장 → 단위 × 레이어수
    - 체크포인팅 OFF: 어텐션·MLP 중간 버퍼까지 모두 보관
                     → 경험적으로 약 M배(≈16배) 더 필요
    """
    # 레이어 하나의 입력 텐서 크기 (바이트)
    unit = batch * seq * HIDDEN * BYTES

    # 체크포인팅 ON/OFF에 따라 저장해야 할 버퍼 배수 결정
    # ON : 레이어 수만큼(경계만 보관, 역전파 때 재계산)
    # OFF: 레이어 수 × M배(중간값 전부 보관)
    mult = LAYERS * (1 if checkpoint else M)

    return unit * mult / GiB  # GiB 단위로 반환


# 비교할 시나리오: (설명, batch 크기, 시퀀스 길이)
# seq 256  ≈ 짧은 질문 + 작은 이미지 토큰
# seq 512  ≈ 중간 길이 시퀀스
# seq 1024 ≈ 긴 질문 또는 고해상도 이미지(이미지 토큰 수백 개)
scenarios = [
    ('batch1 · seq256  (질문+작은 이미지)', 1, 256),
    ('batch2 · seq512',                  2, 512),
    ('batch4 · seq1024 (긴 시퀀스/큰 이미지)', 4, 1024),
]

print(f'{"설정":<34}{"ckpt ON":>12}{"ckpt OFF":>12}')
print('-' * 58)
for name, b, s in scenarios:
    on  = act_gib(b, s, checkpoint=True)   # 체크포인팅 ON  → 활성값 최소화
    off = act_gib(b, s, checkpoint=False)  # 체크포인팅 OFF → 활성값 최대
    print(f'{name:<32}{on:>9.2f} GiB{off:>9.2f} GiB')
print('-' * 58)

# 비례 관계 확인
# batch 1→4(4배), seq 256→1024(4배) 동시 증가 → 활성값 = 4×4 = 16배 예상
ratio = act_gib(4, 1024) / act_gib(1, 256)   # batch·seq 각 4배 → 16배 (체크포인팅과 무관하게 동일)
print(f'batch 1→4, seq 256→1024 (각 4배) ⇒ 활성값 약 {ratio:.0f}배')
# → 활성값이 batch × seq 에 정확히 정비례함을 수치로 확인

설정                                     ckpt ON    ckpt OFF
----------------------------------------------------------
batch1 · seq256  (질문+작은 이미지)         0.04 GiB     0.70 GiB
batch2 · seq512                      0.18 GiB     2.81 GiB
batch4 · seq1024 (긴 시퀀스/큰 이미지)       0.70 GiB    11.25 GiB
----------------------------------------------------------
batch 1→4, seq 256→1024 (각 4배) ⇒ 활성값 약 16배


### 출력 읽기 — 표의 세 줄과 마지막 배수

- `batch1·seq256` (질문+작은 이미지): ON **0.04** / OFF **0.70 GiB** — 작은 설정에선 활성값이 거의 부담이 안 됩니다.
- `batch2·seq512`: ON **0.18** / OFF **2.81 GiB**.
- `batch4·seq1024` (긴 시퀀스/큰 이미지): ON **0.70** / OFF **11.25 GiB** — OFF는 *활성값만으로* T4(16GB)를 위협합니다(가중치까지 더하면 초과).
- 모든 행에서 **OFF ÷ ON ≈ 16배로 일정** — 이 16배가 바로 *checkpointing이 재계산으로 아끼는 양*(어텐션·MLP 중간 버퍼)입니다.
- 마지막 줄: batch·seq를 각 4배로 키우니 활성값이 **16배**(=4×4). 활성값이 **batch·seq에 정비례**함을 수치로 보여줍니다.

**그래서 무엇을 알 수 있나**
- **체크포인팅 OFF**의 *batch4·seq1024* 는 활성값만 **10 GiB가 넘어**, 가중치까지 더하면 T4가 곧장 터집니다.
- **체크포인팅 ON**이면 같은 설정도 **1 GiB 미만** — 그래서 Day1-4에서 grad checkpointing을 켭니다.
- T4에서 OOM이 나면 **batch를 줄이고, 이미지 해상도(=이미지 토큰=seq)를 낮추는 게** 효과적인 이유가 이 정비례 관계에 있습니다.

> 정리: 5-2의 분해 합계는 **활성값을 뺀, 하한에 가까운 값**입니다. 실제 학습 피크 = (그 합계) + (위 활성값). 변동이 큰 활성값까지 합한 **실측 피크 VRAM은 Day1-4에서 직접** 확인합니다. 그리고 네 구성을 줄이는 담당 기법이 다릅니다 — *가중치·그래디언트·옵티마이저*는 4bit·LoRA·8bit 옵티마이저가, ***활성값은 gradient checkpointing***이 맡습니다.

** 이미지가 클수록 activateion 폭주 - 학습 필요

## 6. 기법 지도 — PTQ · QAT · QLoRA · AWQ는 어디에?

양자화 기법은 **언제 양자화하느냐**로 나뉩니다.

| 기법 | 분류 | 언제 | 학습 필요? | 이 과정 |
|---|---|---|---|---|
| **QLoRA** | 학습 시 4bit + 어댑터 학습 | 파인튜닝 중 | ✅ (어댑터) | **Day 1** |
| **PTQ** | 학습 후 양자화 (Post-Training) | 학습 끝난 모델에 | ❌ (소수 calib만) | (개념) |
| **AWQ** | PTQ의 한 종류 (activation-aware) | 학습 끝난 모델에 | ❌ (calibration) | **Day 2** |
| QAT | 양자화를 흉내내며 학습 | 학습 중 | ✅ (전체) | (비교만) |

- **QLoRA**(Day1): *학습*을 가볍게. 4bit 동결 base + LoRA 학습.
- **AWQ**(Day2): *배포*를 가볍게. 이미 학습된 모델을 소수 calibration 데이터로 4bit 변환 → **vLLM**으로 서빙.
- VLM에서 AWQ는 **LLM 선형층만** 양자화하고 **비전 타워·프로젝터는 보존**합니다(지각 품질 유지) — Day2-1에서 자세히.

---

### 다음 교시 예고 — Day1-2 · 환경·4bit 로드·베이스라인
`BitsAndBytesConfig(nf4)`로 **Qwen3-VL-4B를 T4에 4bit로 직접 올리고**, VRAM을 측정하고, 파인튜닝 **전** ChartQA 추론을 관찰해 학습의 기준점을 잡습니다.

> ✅ **이 교시 체크포인트**: ① 경량화 두 갈래(학습/배포)를 말로 설명할 수 있다 ② 8B fp16이 왜 T4에 안 들어가는지 수치로 안다 ③ QLoRA가 무엇을 4bit로 고정하고 무엇을 학습하는지 안다.